--Read Employee Bronze--

In [0]:
from src.constants import *
from src.common_functions import *
from src.validations import *

employee_df = spark.table(EMPLOYEE_BRONZE_TABLE)

In [0]:
employee_df.printSchema()

--Select Facility columns--

In [0]:
facility_df = employee_df.select(
    "Facility_Code",
    "facname",
    "ingestion_timestamp"
)

In [0]:
facility_df = facility_df.withColumnRenamed(
    "facname",
    "Facility_Name"
)

In [0]:
from pyspark.sql.functions import col

facility_df = facility_df.withColumn(
    "Facility_Code",
    col("Facility_Code").cast("int")
)

--Null Validation--

In [0]:
facility_df = facility_df.filter(col("Facility_Code").isNotNull())

print("Facility records after cleaning:", facility_df.count())



--Remove duplicate facilities--

In [0]:
facility_df = (
    facility_df
    .orderBy(
        F.col("ingestion_timestamp").desc()
    )
    .dropDuplicates(["Facility_Code"])
)

In [0]:
print("Unique Facility records:", facility_df.count())

In [0]:
from pyspark.sql.functions import col

facility_df = facility_df.withColumn(
    "Facility_Code",
    col("Facility_Code").cast("int")
)

In [0]:
import importlib
import src.constants as constants

importlib.reload(constants)

print(constants.FACILITY_SILVER_TABLE)

--Save to silver--

In [0]:
from pyspark.sql import functions as F

FACILITY_SILVER_TABLE = "databricks_project1.silver.facility"

# Existing Silver Facility
facility_silver_df = spark.table(
    FACILITY_SILVER_TABLE
)

# Find Facility codes that don't already exist
new_facility_df = (
    facility_df.alias("src")
    .join(
        facility_silver_df.select(
            "Facility_Code"
        ).alias("tgt"),
        on="Facility_Code",
        how="left_anti"
    )
)

new_facility_count = new_facility_df.count()

print(
    "New Facility records:",
    new_facility_count
)

if new_facility_count == 0:

    print(
        "No new Facility records to load. "
        "Silver Facility table remains unchanged."
    )

else:

    new_facility_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(FACILITY_SILVER_TABLE)

    print(
        f"New Facility records loaded: {new_facility_count}"
    )

In [0]:
facility_silver_df = spark.table(
    "databricks_project1.silver.facility"
)

print(
    "Final Silver Facility count:",
    facility_silver_df.count()
)

facility_silver_df.printSchema()

